# Boeing BCAI - Basic Chat + Embedding Call
### VS Code / local Jupyter edition - `BoeingChatModel` + `BoeingEmbeddings` (no direct OpenAI calls)

Same two calls as the Colab version, adapted for running locally in VS Code's Jupyter extension:

1. One chat call via `BoeingChatModel`
2. One embedding call via `BoeingEmbeddings` (`text-embedding-3-small`)

**Local setup differences from Colab:**
- Token comes from a `.env` file (via `python-dotenv`), not Colab Secrets.
- `boeing_chat_model.py` / `boeing_embeddings.py` are expected to sit next to this notebook, not uploaded at runtime.
- Dependencies are installed once into your local/virtual environment ahead of time, not via a `!pip install` cell.

**Before running:** create a `.env` file in the same folder as this notebook:

```
UDAL_PAT=your_actual_token_here
```

Then select that environment's kernel in VS Code (Command Palette -> "Jupyter: Select Interpreter to Start Jupyter Server").


## 0. Setup

Install once in your terminal (not as a notebook cell):

```bash
pip install langchain-core pydantic httpx requests python-dotenv
```


In [ ]:
import os
from pathlib import Path

required = ["boeing_chat_model.py", "boeing_embeddings.py"]
missing = [f for f in required if not Path(f).exists()]
assert not missing, (
    f"Missing: {missing}. Place these files in the same folder as this notebook before continuing."
)
print("Wrapper files present:", required)


In [ ]:
from dotenv import load_dotenv

# Loads variables from a .env file in the same folder as this notebook into the environment.
load_dotenv()

UDAL_PAT = os.getenv("UDAL_PAT")
assert UDAL_PAT, "Set UDAL_PAT in a .env file in this folder before continuing."
print("UDAL_PAT loaded:", UDAL_PAT[:4] + "..." + UDAL_PAT[-4:])


## 1. Boeing API call (chat)

In [ ]:
from boeing_chat_model import BoeingChatModel
from langchain_core.messages import HumanMessage

llm = BoeingChatModel(udal_pat=UDAL_PAT, model="gpt-4.1-mini", temperature=0.2, max_tokens=200)

response = llm.invoke([HumanMessage(content="Say hello in one short sentence.")])

print(response.content)
print("Token usage:", response.response_metadata.get("usage"))


## 2. Boeing embedding call (`text-embedding-3-small`)

In [ ]:
from boeing_embeddings import BoeingEmbeddings

embeddings = BoeingEmbeddings(udal_pat=UDAL_PAT, model="text-embedding-3-small")

vector = embeddings.embed_query("Hello from Boeing BCAI.")

print("Embedding dimension:", len(vector))
print("First 5 values:", vector[:5])
